# Overview of the Encoder Architecture

![encoder_overview](../resources/v1_0/encoder_overview.png)

The encoder architecture is transformer-based with linear attention, SwiGLU and RMSNorm. As part of the encoder we also have a second masked predictor that processes the output of the masked context encoder during training. This notebook will walk through each layer so that the reader can build an intuition for what each layer is doing to the data. To that end, you'll see that we set the layer initializations and numbers to simple values so you can calculate them by hand if you need to follow a layer more closely.

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import math

## Data Prep

We'll start with a simple data prep. Here we'll use a small batch of 2 samples, each with 9 gene expression counts. 

In [2]:
batch = 2 # Batch
num_genes = 9 # context, aka num of genes


SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
x_input = torch.from_numpy(np.round(np.random.uniform(1, 5, size=(batch, num_genes)), 0)).float() # [batch, num_genes]
raw_total_counts = torch.tensor([8_000.0, 12_000.0])
batch_mean_total = torch.tensor([10_000.0, 10_000.0])
total_counts = torch.log1p(raw_total_counts / batch_mean_total) # [batch]

x_input.shape, x_input, total_counts.shape, total_counts

(torch.Size([2, 9]),
 tensor([[2., 2., 2., 3., 2., 3., 2., 5., 4.],
         [1., 3., 4., 2., 5., 3., 4., 4., 2.]]),
 torch.Size([2]),
 tensor([0.5878, 0.7885]))

## Unknown Gene Handling

Our datasets don't always have every gene. Because of this, we need to have a way of handling when a gene isn't in our training data differently than when a gene is 0 count or masked. To handle this, during data prep we create a mask to flag which of the 10k genes a sample has. For now, we'll stage a mask and show how each sample can have a different mask.

In [4]:
unknown_mask = torch.tensor([
    [False, True, False, False, False, False, False, False, True],
    [False, False, True, False, False, False, False, False, False]
])
unknown_mask.shape, unknown_mask

(torch.Size([2, 9]),
 tensor([[False,  True, False, False, False, False, False, False,  True],
         [False, False,  True, False, False, False, False, False, False]]))

## Data Masking 
(only done for the Context/Student) The self-supervised learning is done via masked prediction where we create a mask to make the target prediction task harder. During loss, we primarily start by evaluating the masked positions and slowly add in the rest (excluding unknown). We'll first generate a random distribution and then everything below our target masking threshold will be masked. For the sake of the demonstration I'll use a lower masking ratio than our model.

We apply the masking twice, before our forward pass and after the input projection. We first mask the raw expression values before they enter the encoder to prevent the expression value from influencing the FiLM conditioning (the Fourier features and gamma/beta) at masked positions. Without this, the nonlinear FiLM path would be influenced by the masked positions even though the position gets replaced later. The unknown positions are different as they don't contain real info, so there's no information to leak.

In [5]:
mask_ratio = 0.4 
rand = torch.rand(batch, num_genes)
rand

tensor([[0.0783, 0.4956, 0.6231, 0.4224, 0.2004, 0.0287, 0.5851, 0.6967, 0.1761],
        [0.2595, 0.7086, 0.5809, 0.0574, 0.7669, 0.8778, 0.2434, 0.6005, 0.7079]])

In [6]:
mask_idx = rand < mask_ratio  # use probabilistic masking; it works well over a large training run
mask_idx

tensor([[ True, False, False, False,  True,  True, False, False,  True],
        [ True, False, False,  True, False, False,  True, False, False]])

In [7]:
x_values = x_input
x_values[mask_idx] = 0.0
x_values

tensor([[0., 2., 2., 3., 0., 0., 2., 5., 0.],
        [0., 3., 4., 0., 5., 3., 0., 4., 2.]])

## Forward Pass

We start by adding a channel dimension. Right now we just have 1 value per gene, but we'll represent each gene with many dimensions `embed_dim` to let the model learn different combinations of gene importance. We'll also use multiple `heads`, which are groupings of the embedding dimension channels, so that combinations of them can learn different complex topics.

In [8]:
embed_dim = 6
heads = 2

**Insert the channel dimension**

In [9]:
x = x_values.unsqueeze(-1)
x.shape, x

(torch.Size([2, 9, 1]),
 tensor([[[0.],
          [2.],
          [2.],
          [3.],
          [0.],
          [0.],
          [2.],
          [5.],
          [0.]],
 
         [[0.],
          [3.],
          [4.],
          [0.],
          [5.],
          [3.],
          [0.],
          [4.],
          [2.]]]))

### Fourier FiLM gene encoding

Instead of just relying on gene expression counts, we want to do a Fourier FiLM based projection of the gene expression counts with learnable controls on the projection. 

![full_model_overview](../resources/v1_0/fourier_film_encoder.png)

We do this since we know that in biology, expression is typically non-linear where you have different expression plateaus including fully on or off. In this projection expression values are first scaled by learned gene embeddings and a scaler, then a random Fourier feature network generates expression-dependent FiLM parameters (gamma, beta) to further modulate the result based on expression level. The two paths give the model both a linear signal (expression * embedding) and a nonlinear one (Fourier features -> FiLM), combined into the final gene representation. The goal is to apply the following

$h_g = \alpha x_g e_g \odot (1 + \gamma_g) + \beta_g$

where

$[\gamma_g, \beta_g] = \mathrm{MLP}(\phi(\alpha' x_g))$

$x_g$ is the expression value for gene $g$ while $e_g$ is the learned gene embedding, $\alpha, \alpha'$ are learned scalars and $\phi$ is a random Fourier feature mapping 

While we use a multilayer perceptron (MLP), the FiLM portion is specifically the $\mathbf{h}_g = x * (1 + \gamma) + \beta$ pattern. This becomes an affine transformation where gamma and beta are conditioned on some input. 

You might notice that if we simply did this, we'd lose the gene identity for true zero-count genes in our data. After we calculate the projection, we then add back the gene identity as a residual so that after the FiLM modulation, each gene that exists in the dataset retains a baseline gene identity signal regardless of expression level. This allows the model to build different influences on the gene embeddings for "gene exists" and "gene expressed by this much."

#### $\alpha$ Gene Expression Count Scaling

We'll start with our gene expression count scaler. This will be a single value that we'll multiply against our scaled gene expression count embeddings. We use a linear layer here so that backprop can update this scaler. We'll first set up the learned scaler and multiply it by expression counts, after which we'll then use that to scale our gene embeddings. We'll initialize this scaler to `1.5` so you'll see that our initial expression values grow.

In [10]:
expr_scaler = nn.Linear(1, 1, bias=False)
nn.init.constant_(expr_scaler.weight, 1.5)
expr_scaler.weight

Parameter containing:
tensor([[1.5000]], requires_grad=True)

In [11]:
scaled_x = expr_scaler(x)
scaled_x.shape, scaled_x

(torch.Size([2, 9, 1]),
 tensor([[[0.0000],
          [3.0000],
          [3.0000],
          [4.5000],
          [0.0000],
          [0.0000],
          [3.0000],
          [7.5000],
          [0.0000]],
 
         [[0.0000],
          [4.5000],
          [6.0000],
          [0.0000],
          [7.5000],
          [4.5000],
          [0.0000],
          [6.0000],
          [3.0000]]], grad_fn=<UnsafeViewBackward0>))

#### $\mathbf{e}_g$ Gene Embeddings

Now we'll initialize a representation of the gene embeddings and then scale them based on our scaled expression counts. You'll see that the masking extends across all embedding channels. Also, since we used an initialization where each channel has the same value, you'll see that the multiplication creates consistent channel entries per gene.

In [12]:
# creates an incremental weight for easier following
vs, d = num_genes, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1*(rows + cols)  

gene_embeddings = nn.Parameter(pattern)
gene_embeddings

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000],
        [0.7000, 0.7000, 0.7000, 0.7000, 0.7000, 0.7000],
        [0.8000, 0.8000, 0.8000, 0.8000, 0.8000, 0.8000],
        [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000]], requires_grad=True)

In [13]:
scaled_x = gene_embeddings.unsqueeze(0) * scaled_x
scaled_x

tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000],
         [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
         [1.8000, 1.8000, 1.8000, 1.8000, 1.8000, 1.8000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [2.1000, 2.1000, 2.1000, 2.1000, 2.1000, 2.1000],
         [6.0000, 6.0000, 6.0000, 6.0000, 6.0000, 6.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
         [1.8000, 1.8000, 1.8000, 1.8000, 1.8000, 1.8000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [3.7500, 3.7500, 3.7500, 3.7500, 3.7500, 3.7500],
         [2.7000, 2.7000, 2.7000, 2.7000, 2.7000, 2.7000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [4.8000, 4.8000, 4.8000, 4.8000, 4.8000, 4.80

#### $\alpha'$ Fourier Scaler 

Now we move on to the Fourier half of the calculation. We'll start by calculating the scaler. Again we use a linear here so that it represents a learnable parameter that will drift during backprop. To make sure the values diverge from the expression scaler, I'll use a different value `0.5`. When applied, you can see our initial expression values are cut in half. 

In [14]:
fourier_input_scaler = nn.Linear(1, 1, bias=False)
nn.init.constant_(fourier_input_scaler.weight, 0.5)
fourier_input_scaler.weight

Parameter containing:
tensor([[0.5000]], requires_grad=True)

In [15]:
fourier_x = fourier_input_scaler(x)
fourier_x.shape, fourier_x

(torch.Size([2, 9, 1]),
 tensor([[[0.0000],
          [1.0000],
          [1.0000],
          [1.5000],
          [0.0000],
          [0.0000],
          [1.0000],
          [2.5000],
          [0.0000]],
 
         [[0.0000],
          [1.5000],
          [2.0000],
          [0.0000],
          [2.5000],
          [1.5000],
          [0.0000],
          [2.0000],
          [1.0000]]], grad_fn=<UnsafeViewBackward0>))

#### $\phi$ Fourier Projection
Now we'll do the Fourier projection. This step projects the scaled expression counts through a fixed random matrix `fp_scaler`. This random matrix is half the size since we take the sin and cos of the result and then concatenate them together to produce a full-dimensional embedding. 

This step maps the continuous expression level into a rich high-dimensional representation where nearby values have similar features allowing the model to learn representations from the different plateaus of expression levels. The `gaussian_scale` scaler is a tunable hyperparameter that can quickly improve and degrade model performance.

Since we're working with sine and cosine, we need to express our sampled frequencies as angles in radians. We multiply the expression value by $2\pi$ before applying the fixed Fourier projection. The values in `fp_scaler` determine how quickly each channel cycles as expression changes, while `gaussian_scale` controls the overall spread of those frequencies. The $2\pi$ converts those sampled frequencies into the angular values expected by sine and cosine.

**Numeric computing** One thing to remember is that we're working with finite precision. Because values such as $\pi$ cannot be represented exactly, a sine or cosine value that should theoretically be zero may appear as a very small nonzero number. This is normal deterministic floating-point approximation and does not change the logic of the forward pass.

In [16]:
examp_array = torch.tensor([0,0.25,0.5,0.75,1]).float()
torch.sin(examp_array*2*np.pi), torch.cos(examp_array*2*np.pi)

(tensor([ 0.0000e+00,  1.0000e+00, -8.7423e-08, -1.0000e+00,  1.7485e-07]),
 tensor([ 1.0000e+00, -4.3711e-08, -1.0000e+00,  1.1925e-08,  1.0000e+00]))

In [17]:
x_fp = (2 * np.pi * fourier_x)
x_fp

tensor([[[ 0.0000],
         [ 6.2832],
         [ 6.2832],
         [ 9.4248],
         [ 0.0000],
         [ 0.0000],
         [ 6.2832],
         [15.7080],
         [ 0.0000]],

        [[ 0.0000],
         [ 9.4248],
         [12.5664],
         [ 0.0000],
         [15.7080],
         [ 9.4248],
         [ 0.0000],
         [12.5664],
         [ 6.2832]]], grad_fn=<MulBackward0>)

Now that we've scaled our expression counts by $2\pi$, we'll inject half of the channels. As part of this injection, we use fixed randomly initialized frequencies for each channel and a tunable hyperparameter `gaussian_scale` to control their spread. For learning, we'll keep our initialization preset.

That said, you'll see that we do not use gradients as this is not learnable. Our goal with the random Fourier features is that a fixed random projection is theoretically sufficient to approximate a shift-invariant kernel (we care about differences in expression values, not the actual numeric number). If we made it learnable, the network would likely collapse the frequencies to overfit or degenerate, defeating the purpose of providing a diverse multi-frequency basis. The nonlinear expressiveness of this layer comes downstream with the FiLM generator MLP that processes the Fourier features. That layer is learnable.

In [18]:
gaussian_scale = 2.0
fp_scaler = nn.Parameter(torch.tensor([[0.5,1.0,1.5]]) * gaussian_scale, requires_grad=False)
fp_scaler

Parameter containing:
tensor([[1., 2., 3.]])

In [19]:
x_fp = x_fp @ fp_scaler
x_fp.shape, x_fp

(torch.Size([2, 9, 3]),
 tensor([[[ 0.0000,  0.0000,  0.0000],
          [ 6.2832, 12.5664, 18.8496],
          [ 6.2832, 12.5664, 18.8496],
          [ 9.4248, 18.8496, 28.2743],
          [ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 6.2832, 12.5664, 18.8496],
          [15.7080, 31.4159, 47.1239],
          [ 0.0000,  0.0000,  0.0000]],
 
         [[ 0.0000,  0.0000,  0.0000],
          [ 9.4248, 18.8496, 28.2743],
          [12.5664, 25.1327, 37.6991],
          [ 0.0000,  0.0000,  0.0000],
          [15.7080, 31.4159, 47.1239],
          [ 9.4248, 18.8496, 28.2743],
          [ 0.0000,  0.0000,  0.0000],
          [12.5664, 25.1327, 37.6991],
          [ 6.2832, 12.5664, 18.8496]]], grad_fn=<UnsafeViewBackward0>))

**Sin/Cos**

Now we're ready to take the sine and cosine. As a reminder, because we're dealing with numeric computing, instead of nice clean integers we'll see that we have infinitesimals introduced. This is more noticeable for sine than cosine. Concatenating the two outputs brings us back to our full embedding dimension.

In [20]:
x_fp_sin = torch.sin(x_fp)
x_fp_sin

tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [-6.7553e-07,  1.3511e-06, -3.9339e-06],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00]],

        [[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 3.4969e-07,  6.9938e-07,  9.5399e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [-6.7553e-07,  1.3511e-06, -3.9339e-06],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 3.4969e-07,  6.9938e-07,  9.5399e-08],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08]]], grad_fn=<SinBackward0>)

In [21]:
x_fp_cos = torch.cos(x_fp)
x_fp_cos

tensor([[[ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.]],

        [[ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.]]], grad_fn=<CosBackward0>)

In [22]:
fourier_x = torch.cat([x_fp_sin, x_fp_cos], dim=-1)
fourier_x.shape, fourier_x

(torch.Size([2, 9, 6]),
 tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [-2.3850e-08,  4.7700e-08, -7.1549e-08, -1.0000e+00,  1.0000e+00,
           -1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [-6.7553e-07,  1.3511e-06, -3.9339e-06, -1.0000e+00,  1.0000e+00,
           -1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00]],
 
         [[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  

#### $\mathrm{MLP}$ Multilayer perceptron

Now we're ready to add nonlinear expressiveness to the FiLM generator by using an MLP. The MLP provides a learnable linear layer, a nonlinearity, and a final projection that doubles the dimension to create our $\gamma$ and $\beta$ for the FiLM calculation. These learnable layers are what allow the model to decide how much and which parts of the Fourier projection we want to include with our initial scaled expression.

**MLP - linear 1** We'll first start with a single linear layer that allows the model to decide how much each channel should interact with the others. Recall that half our channels are sine and half are cosine, so this allows the model to mix the two radial projections.

*You'll notice that near 0 values here are washed out*

In [23]:
mlp_l1 = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(mlp_l1.weight, .25)
nn.init.constant_(mlp_l1.bias, 0.0)
mlp_l1.weight

Parameter containing:
tensor([[0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500]], requires_grad=True)

In [24]:
fourier_x = mlp_l1(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 9, 6]),
 tensor([[[ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500]],
 
         [[ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0

**MLP - GELU nonlinearity** Now we're ready for our non-linearity. The [GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html) function is approximately linear above 1 and pulls most values below -2 to 0. Between -2 and 0, most values are pulled closer to 0 and there's a slight non-linearity between 0 and 1. Since we used sine/cosine values, most of our values at this point should be between 0 and 1 so we'll benefit from the non-linear portion with a cap on highly negative values. 

In [25]:
mlp_gelu = nn.GELU()

fourier_x = mlp_gelu(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 9, 6]),
 tensor([[[ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800]],
 
         [[ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0

**MLP - linear 2** Now we'll scale up to 2x our embedding size since we need to provide values for our two variables in our FiLM projection. We use a learnable linear scale up layer so that the model can learn how to split values across the two variables. For our initialization we'll increment the second half to be twice the first half to show the differences. 

In [26]:
mlp_l2 = nn.Linear(embed_dim, embed_dim*2)
nn.init.constant_(mlp_l2.weight[:embed_dim, :], 0.5)
nn.init.constant_(mlp_l2.weight[embed_dim:, :], 1.0)
nn.init.constant_(mlp_l2.bias, 0.0)
mlp_l2.weight.shape, mlp_l2.weight

(torch.Size([12, 6]),
 Parameter containing:
 tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]], requires_grad=True))

In [27]:
fourier_x = mlp_l2(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 9, 12]),
 tensor([[[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.6019,
           -0.6019, -0.6019, -0.6019, -0.6019, -0.6019],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0

#### $\gamma_g, \beta_g$ FiLM components 
Now that we have the MLP output, we're ready to build our FiLM variables. Recall that for FiLM we calculate $\mathbf{h}_g = x * (1 + \gamma) + \beta$ where $x$ will be the weighted scaler of the expression counts. To get $\gamma$ and $\beta$ we simply split the output of the MLP. Recall that we built it so that half of the MLP output was doubled so when we split, we should see that $\beta$ is double $\gamma$.

In [28]:
gamma, beta = torch.chunk(fourier_x, 2, dim=-1)
gamma.shape, gamma, beta.shape, beta

(torch.Size([2, 9, 6]),
 tensor([[[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401]],
 
         [[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0

#### $\mathbf{h}_g$ FiLM based gene expression representation 

Now we're ready for the final FiLM calculation. FiLM can be thought of as summing two weighted parts, in our case a scaled version of the expression counts and a radial projection of the expression counts. What's interesting is that the FiLM formula pushes the radial projection both as a scaler to the initial counts and a bias similar to $x = mx + b$. Ultimately we've given the model the ability to learn how to upscale the original counts, shift the counts based on the radial projection, and add/subtract the counts based on the radial projection. All of this allows the model to learn a more complex landscape to scale gene embeddings by the expression beyond the typical linear scaling where 2 expression counts mean double 1 expression. 

In [29]:
x = scaled_x * (1.0 + gamma) + beta

x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 5.1242,  5.1242,  5.1242,  5.1242,  5.1242,  5.1242],
          [ 5.9463,  5.9463,  5.9463,  5.9463,  5.9463,  5.9463],
          [ 0.6563,  0.6563,  0.6563,  0.6563,  0.6563,  0.6563],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 9.2344,  9.2344,  9.2344,  9.2344,  9.2344,  9.2344],
          [ 3.5922,  3.5922,  3.5922,  3.5922,  3.5922,  3.5922],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802]],
 
         [[ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 0.0272,  0.0272,  0.0272,  0.0272,  0.0272,  0.0272],
          [ 8.4123,  8.4123,  8.4123,  8.4123,  8.4123,  8.4123],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 2.0194,  2.0194,  2.0194,  2.0194,  2.0194,  2.0194],
          [ 1.2854,  1.2854,  1.2854,  1.2854,  1

#### Residual Gene Embedding

Now we'll add our gene embedding back to the expression-based FiLM projection value. As mentioned before, this ensures that the model learns the difference between a gene existing and not existing. If we just let the 0 count project through, we'd collapse all the 0 count genes down to a single representation, confusing the model further.

In [30]:
x = gene_embeddings.unsqueeze(0) + x
x

tensor([[[ 3.5802,  3.5802,  3.5802,  3.5802,  3.5802,  3.5802],
         [ 5.3242,  5.3242,  5.3242,  5.3242,  5.3242,  5.3242],
         [ 6.2463,  6.2463,  6.2463,  6.2463,  6.2463,  6.2463],
         [ 1.0563,  1.0563,  1.0563,  1.0563,  1.0563,  1.0563],
         [ 3.9802,  3.9802,  3.9802,  3.9802,  3.9802,  3.9802],
         [ 4.0802,  4.0802,  4.0802,  4.0802,  4.0802,  4.0802],
         [ 9.9344,  9.9344,  9.9344,  9.9344,  9.9344,  9.9344],
         [ 4.3922,  4.3922,  4.3922,  4.3922,  4.3922,  4.3922],
         [ 4.3802,  4.3802,  4.3802,  4.3802,  4.3802,  4.3802]],

        [[ 3.5802,  3.5802,  3.5802,  3.5802,  3.5802,  3.5802],
         [ 0.2272,  0.2272,  0.2272,  0.2272,  0.2272,  0.2272],
         [ 8.7123,  8.7123,  8.7123,  8.7123,  8.7123,  8.7123],
         [ 3.8802,  3.8802,  3.8802,  3.8802,  3.8802,  3.8802],
         [ 2.5194,  2.5194,  2.5194,  2.5194,  2.5194,  2.5194],
         [ 1.8854,  1.8854,  1.8854,  1.8854,  1.8854,  1.8854],
         [ 4.1802,  4.1

#### Remask with learned mask

Now we need to reintroduce the masking but, instead of a 0 value, we want to let the model learn a mask token. Previously we ensured the expression values were hidden for the expression count projection, while this time we create a learnable mask signal for the positions to tell the model to fill them in. We learn a mask token because the model needs to distinguish "this gene is masked and I need to predict it" from "this gene has zero expression." If the mask token were fixed (e.g. all zeros), it would be indistinguishable from a zero-expression gene's representation after the Fourier FiLM encoding. A learned token lets the model settle on a representation that optimally signals "predict me" to the downstream transformer blocks. Ultimately we're reapplying the masking at this point mainly so that we do our expression encoding cleanly first, and then mask after. Since our mask token is learnable, instead of a common `-1` hardcode, the initialization in the model will use random learnable values. What we'll do in our example is use `-11` so it really sticks out. These values will change during backprop as the model learns what a good token is to prompt it that it needs replacement.

*As a reminder, masking only happens on the context encoder, and not the target encoder during our model's forward pass*

In [31]:
mask_token = nn.Parameter(torch.randn(embed_dim) * 0.02)
nn.init.constant_(mask_token, -11)
mask_token

Parameter containing:
tensor([-11., -11., -11., -11., -11., -11.], requires_grad=True)

In [32]:
x = torch.where(mask_idx.unsqueeze(-1), mask_token, x)
x

tensor([[[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  5.3242,   5.3242,   5.3242,   5.3242,   5.3242,   5.3242],
         [  6.2463,   6.2463,   6.2463,   6.2463,   6.2463,   6.2463],
         [  1.0563,   1.0563,   1.0563,   1.0563,   1.0563,   1.0563],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  9.9344,   9.9344,   9.9344,   9.9344,   9.9344,   9.9344],
         [  4.3922,   4.3922,   4.3922,   4.3922,   4.3922,   4.3922],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000]],

        [[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  0.2272,   0.2272,   0.2272,   0.2272,   0.2272,   0.2272],
         [  8.7123,   8.7123,   8.7123,   8.7123,   8.7123,   8.7123],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  2.5194,   2.5194,   2.5194,   2.5194,   2.5194,   2.5194],
    

#### Mask unknown genes with learnable unknown token

Now we need to replace the unknown gene positions with a learnable unknown token. An unknown gene is different from a zero-count gene: a zero means the gene was measured but had no observed expression, while unknown means the gene was not measured in this dataset.

The same unknown token replaces the gene-specific representation at each unknown position, giving the encoder a consistent signal that the expression value is unavailable. These positions still participate in attention like the other positions. Separately, the gene mask is used during loss so that we do not score reconstruction at genes we never measured. For our example, we'll use `-5` so the unknown token is easy to distinguish from the masked positions.

*As a reminder, unknown masking happens in both the student and teacher encoders*

In [33]:
unknown_token = nn.Parameter(torch.randn(embed_dim) * 0.02)
nn.init.constant_(unknown_token, -5)
unknown_token

Parameter containing:
tensor([-5., -5., -5., -5., -5., -5.], requires_grad=True)

In [34]:
x = torch.where(unknown_mask.unsqueeze(-1), unknown_token, x)
x

tensor([[[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [ -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000],
         [  6.2463,   6.2463,   6.2463,   6.2463,   6.2463,   6.2463],
         [  1.0563,   1.0563,   1.0563,   1.0563,   1.0563,   1.0563],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  9.9344,   9.9344,   9.9344,   9.9344,   9.9344,   9.9344],
         [  4.3922,   4.3922,   4.3922,   4.3922,   4.3922,   4.3922],
         [ -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000]],

        [[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  0.2272,   0.2272,   0.2272,   0.2272,   0.2272,   0.2272],
         [ -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  2.5194,   2.5194,   2.5194,   2.5194,   2.5194,   2.5194],
    

### Total count injection

Now we'll want to add the total count feature. During data prep, we normalize and log-transform the expression values, which removes the original total expression signal from `x_values`. To retain that information, we calculate the cell's raw total relative to the mean total for its batch and then compress it with `log1p`:

$$t = \log\left(1 + \frac{\text{raw cell total}}{\text{batch mean total}}\right)$$

This gives the encoder a batch-relative signal for whether the cell had unusually high or low total expression without feeding the much larger raw count directly into the model.

The total count is multiplied across the embedding dimensions with a learnable weight and then added to our gene expression projections. This allows the total count to act almost like a bias term.

We start by taking our total count, injecting the embedding dimensions, and then shaping it to match our gene expression representation.

In [35]:
x_total_ct = total_counts.unsqueeze(-1)
x_total_ct.shape, x_total_ct

(torch.Size([2, 1]),
 tensor([[0.5878],
         [0.7885]]))

In [36]:
total_count_proj = nn.Linear(1, embed_dim)
nn.init.constant_(total_count_proj.weight, 0.1)
nn.init.zeros_(total_count_proj.bias)
total_count_proj.weight

Parameter containing:
tensor([[0.1000],
        [0.1000],
        [0.1000],
        [0.1000],
        [0.1000],
        [0.1000]], requires_grad=True)

In [37]:
x_total_ct = total_count_proj(x_total_ct)
x_total_ct = x_total_ct.unsqueeze(1)
x_total_ct.shape, x_total_ct

(torch.Size([2, 1, 6]),
 tensor([[[0.0588, 0.0588, 0.0588, 0.0588, 0.0588, 0.0588]],
 
         [[0.0788, 0.0788, 0.0788, 0.0788, 0.0788, 0.0788]]],
        grad_fn=<UnsqueezeBackward0>))

### Unified representation of the cell state

Now that we have an embedding representation of both the gene expression and total count, we're ready to sum them for a single representation of the cell state. Now it will be ready for a cell state block.

In [38]:
x = x + x_total_ct
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-10.9412, -10.9412, -10.9412, -10.9412, -10.9412, -10.9412],
          [ -4.9412,  -4.9412,  -4.9412,  -4.9412,  -4.9412,  -4.9412],
          [  6.3050,   6.3050,   6.3050,   6.3050,   6.3050,   6.3050],
          [  1.1151,   1.1151,   1.1151,   1.1151,   1.1151,   1.1151],
          [-10.9412, -10.9412, -10.9412, -10.9412, -10.9412, -10.9412],
          [-10.9412, -10.9412, -10.9412, -10.9412, -10.9412, -10.9412],
          [  9.9931,   9.9931,   9.9931,   9.9931,   9.9931,   9.9931],
          [  4.4510,   4.4510,   4.4510,   4.4510,   4.4510,   4.4510],
          [ -4.9412,  -4.9412,  -4.9412,  -4.9412,  -4.9412,  -4.9412]],
 
         [[-10.9212, -10.9212, -10.9212, -10.9212, -10.9212, -10.9212],
          [  0.3060,   0.3060,   0.3060,   0.3060,   0.3060,   0.3060],
          [ -4.9212,  -4.9212,  -4.9212,  -4.9212,  -4.9212,  -4.9212],
          [-10.9212, -10.9212, -10.9212, -10.9212, -10.9212, -10.9212],
          [  2.5983,   2.5983,   2.59

### Transformer Block 
The transformer layer in our model consists of 4 layers:

1. RMS normalization
2. Gated linear self-attention
3. RMS normalization
4. SwiGLU

Residual connections provide gradient bypassing around the attention and SwiGLU layers. This set of 4 units is repeated based on how many layers are configured. 

#### RMSNorm 1

For our modern transformer, we use root mean square normalization, or RMSNorm. RMSNorm calculates the following: 
$$
y = \frac{x}{\sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2 + \epsilon}} \cdot \gamma
$$

The main reason we use RMSNorm is that it executes faster and uses less memory than standard layer normalization. This efficiency is achieved by entirely removing the mean-centering calculation, which reduces the total number of arithmetic operations and hardware synchronization steps. Recall that normalization is primarily used for large scale training stability (preventing gradient explosion/vanishing). For deep architectures like ours, the mean of the pre-activation inputs naturally stays close to zero during training so removing the mean-centering operation preserves the critical variance-bounding effect. Also the model learns to absorb any minor activation shifts into the subsequent linear weights or the learned affine parameters. Because of this, we're able to use a more efficient normalization.

Since we reuse RMSNorm 3 times, we'll create a class for it. In the class you can see that we split out the calculation: first we promote the input to float32 so that the square and reciprocal square root are calculated more reliably, then we normalize by the root mean square and apply the learned per-channel weights. Finally, we cast the result back to the input's original data type.

Since our entries have identical values across all channels for a gene, the gene becomes uniform +/-1 depending on the sign (the value is the average so it becomes 1).

In [39]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        x_fp32 = x.float()
        norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight).type_as(x)

In [40]:
rms1 = RMSNorm(embed_dim)

rms1.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [41]:
x_norm = rms1(x)
x_norm.shape, x_norm

(torch.Size([2, 9, 6]),
 tensor([[[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000]],
 
         [[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1

#### Multi-Headed Gated Linear Attention
Since our current token size is 10000 and we know it's going to grow to 20,000+ when we add all genes and other cell state representations, we want to avoid building the attention matrix in memory. To avoid this we chose linear attention. Linear attention foregoes materializing the attention matrix. In the encoder, we do self attention allowing each gene to build a relationship with any other gene. Our attention will be multi-headed, meaning we'll split up our embedding space across the heads allowing them each to learn different complex representations. Finally we'll also add gating at the end to allow the model to determine how much the attention should impact each specific gene. Linear attention still incorporates the query, key, and value, but removes the need for softmax. We calculate

$$\begin{aligned}
\text{elu}(x) &= \begin{cases} x & \text{if } x > 0 \\ \alpha (e^x - 1) & \text{if } x \le 0 \end{cases} \\
\\
Q &= \text{elu}(x W_q^\top + b_q) + 1.0 \\
K &= \text{elu}(x W_k^\top + b_k) + 1.0 \\
V &= x W_v^\top + b_v \\
\\
\text{Attn} &= \frac{Q K^\top V}{Q (\sum_{j=1}^T K_j)^\top + \epsilon} \\
\text{Gate} &= \sigma(x W_{gate}^\top + b_{gate}) \\
\\
y &= \left( \text{Gate} \odot \text{Attn} \right) W_c^\top + b_c
\end{aligned}$$

In [42]:
B, T_q, C = x_norm.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

(2, 9, 6, 3)

**Self attention** 

For our encoder, the linear attention is self attention, meaning it allows for each token (gene) to build a relationship with the other tokens. Typically this can be extremely memory expensive as we'd make a TxT matrix in memory. As you'll see, with linear attention we do not need to do that. You might now ask: why even include this variable `kv_input`. This is because we built our linear attention to be both self attention and cross attention. Cross attention is when you're building a relationship between two different inputs. 

In [43]:
kv_input = x_norm
kv_input

tensor([[[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000]],

        [[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0

In [44]:
T_kv = kv_input.size(1)
T_kv

9

**Query** 

Let's start with our query. The query is the current position's "search request." Every layer and head issues queries that can look for relationships. Our query first calculates a linear weight $Q=x_{norm}W^\top+b$, resulting in a vector of $[B,T,C]$. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T,C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T,C_{Heads}]$

Finally we'll apply the ELU+1 function. In linear attention, since we're skipping softmax, we need to ensure that the attention weights stay non-negative like they would with softmax. ELU+1 approximates softmax attention's behavior while keeping the linear complexity benefit.

In [45]:
q_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(q_proj.weight, -0.1)
nn.init.constant_(q_proj.bias, 0)
q_proj.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [46]:
q = q_proj(x_norm).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

(torch.Size([2, 2, 9, 3]),
 tensor([[[[ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000]],
 
          [[ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000]]],
 
 
         [[[ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
    

In [47]:
q = F.elu(q) + 1.0
q.shape, q

(torch.Size([2, 2, 9, 3]),
 tensor([[[[1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000]],
 
          [[1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000]]],
 
 
         [[[1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
  

**Key** 

Next, we calculate the key. The key acts as a matching tag/address for each allowed token. It is compared with the query to produce relevance scores. If this was cross-attention and kv was provided, K would be based on it. Since it's self attention, we again project the normalized X allowing the model to build the second half of the cross. 

We again calculate a linear weight $K=x_{kv\_input}\cdot W^\top+b$, resulting in a vector of $[B,T,C]$. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T,C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T,C_{Heads}]$

Finally we'll apply the ELU+1 function. In linear attention, since we're skipping softmax, we need to ensure that the attention weights stay non-negative like they would with softmax. ELU+1 approximates softmax attention's behavior while keeping the linear complexity benefit.

In [48]:
k_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(k_proj.weight, 0.2)
nn.init.constant_(k_proj.bias, 0)
k_proj.weight

Parameter containing:
tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000]], requires_grad=True)

In [49]:
k = k_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

(torch.Size([2, 2, 9, 3]),
 tensor([[[[-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000]],
 
          [[-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000]]],
 
 
         [[[-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
    

In [50]:
k = F.elu(k) + 1.0
k

tensor([[[[0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012]],

         [[0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012]]],


        [[[0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000]],

         [[0.3012, 0

**Value** 

Next, we calculate the value. The value is the payload you actually mix in once something matches. It's a learned projection of the token's representation so the model can copy the right kind of information. Similarly, since this is self attention, V will be based on X. 

We again calculate a linear weight $V=x_{kv\_input}\cdot W^\top+b$, resulting in a vector of $[B,T,C]$. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T,C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T,C_{Heads}]$

For V we do not use ELU+1 since the QK will act on V. 

In [51]:
v_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(v_proj.weight, 1.0)
nn.init.constant_(v_proj.bias, 0)
v_proj.weight

Parameter containing:
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]], requires_grad=True)

In [52]:
v = v_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

(torch.Size([2, 2, 9, 3]),
 tensor([[[[-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000]],
 
          [[-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000]]],
 
 
         [[[-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
    

**Normalization denominator** 

In attention, when using softmax, attention values become probabilities and sum to 1. With linear attention, we need to manually do this normalization.

In softmax attention, the softmax inherently normalizes so weights sum to 1. Linear attention doesn't have that, so we need to create the denominator $z$. We'll do this by summing the tokens per embedding dimension, resulting in a $[B,Heads,C_{Heads},1]$ dimension that we can then use in the denominator of our attention calculation.

In [53]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

(torch.Size([2, 2, 3, 1]),
 tensor([[[[10.3060],
           [10.3060],
           [10.3060]],
 
          [[10.3060],
           [10.3060],
           [10.3060]]],
 
 
         [[[12.2048],
           [12.2048],
           [12.2048]],
 
          [[12.2048],
           [12.2048],
           [12.2048]]]], grad_fn=<UnsqueezeBackward0>))

**Denominator** 

We're now ready to calculate the rest of the denominator. The denominator is the sum of attention weights for each query across all keys. Each query gets its own normalizing constant, so queries attending to high-magnitude keys don't get inflated outputs. We add a final epsilon to ensure the denominator is not zero. The result is a $[B,Heads,T,1]$ tensor.

In [54]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

(torch.Size([2, 2, 9, 1]),
 tensor([[[[0.0202],
           [0.0202],
           [0.0589],
           [0.0589],
           [0.0202],
           [0.0202],
           [0.0589],
           [0.0589],
           [0.0202]],
 
          [[0.0202],
           [0.0202],
           [0.0589],
           [0.0589],
           [0.0202],
           [0.0202],
           [0.0589],
           [0.0589],
           [0.0202]]],
 
 
         [[[0.0171],
           [0.0498],
           [0.0171],
           [0.0171],
           [0.0498],
           [0.0498],
           [0.0171],
           [0.0498],
           [0.0498]],
 
          [[0.0171],
           [0.0498],
           [0.0171],
           [0.0171],
           [0.0498],
           [0.0498],
           [0.0171],
           [0.0498],
           [0.0498]]]], grad_fn=<MulBackward0>))

**Numerator**

Now we're ready to complete our numerator. This is just a matter of multiplying Q, K, and V. We'll need to transpose K to get the interaction between the query and key to then multiply against the value. 

Since we're looking to save memory, we actually first multiply the key and value to create head dimension matrixes, and then multiply by the query. This order of operations is part of what saves memory. 

In [55]:
kv = k.transpose(-2, -1) @ v
kv.shape, kv

(torch.Size([2, 2, 3, 3]),
 tensor([[[[43.7642, 43.7642, 43.7642],
           [43.7642, 43.7642, 43.7642],
           [43.7642, 43.7642, 43.7642]],
 
          [[43.7642, 43.7642, 43.7642],
           [43.7642, 43.7642, 43.7642],
           [43.7642, 43.7642, 43.7642]]],
 
 
         [[[58.7712, 58.7712, 58.7712],
           [58.7712, 58.7712, 58.7712],
           [58.7712, 58.7712, 58.7712]],
 
          [[58.7712, 58.7712, 58.7712],
           [58.7712, 58.7712, 58.7712],
           [58.7712, 58.7712, 58.7712]]]], grad_fn=<UnsafeViewBackward0>))

In [56]:
qkv = q @ kv
qkv.shape, qkv

(torch.Size([2, 2, 9, 3]),
 tensor([[[[210.0680, 210.0680, 210.0680],
           [210.0680, 210.0680, 210.0680],
           [ 72.0548,  72.0548,  72.0548],
           [ 72.0549,  72.0549,  72.0549],
           [210.0680, 210.0680, 210.0680],
           [210.0680, 210.0680, 210.0680],
           [ 72.0548,  72.0548,  72.0548],
           [ 72.0548,  72.0548,  72.0548],
           [210.0680, 210.0680, 210.0680]],
 
          [[210.0680, 210.0680, 210.0680],
           [210.0680, 210.0680, 210.0680],
           [ 72.0548,  72.0548,  72.0548],
           [ 72.0549,  72.0549,  72.0549],
           [210.0680, 210.0680, 210.0680],
           [210.0680, 210.0680, 210.0680],
           [ 72.0548,  72.0548,  72.0548],
           [ 72.0548,  72.0548,  72.0548],
           [210.0680, 210.0680, 210.0680]]],
 
 
         [[[282.1019, 282.1019, 282.1019],
           [ 96.7633,  96.7633,  96.7633],
           [282.1019, 282.1019, 282.1019],
           [282.1019, 282.1019, 282.1019],
           [ 96.76

**Linear Attention** 

Now we're ready to normalize our numerator by the denominator. You'll see that because of our extremely consistent values and initialization we're ending up with very consistent values in this example, even across heads.

In [57]:
y = qkv * z
y.shape, y

(torch.Size([2, 2, 9, 3]),
 tensor([[[[4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465]],
 
          [[4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465]]],
 
 
         [[[4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
  

**Collapse heads** 

We now can bring our heads back together. We have to undo our head splitting. First we flip our heads and tokens (genes) so that we have a $[B,T,Heads,C_{Heads}]$, and then we collapse the head and head embeddings to end up with $[B,T,C]$.

In [58]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

(torch.Size([2, 9, 6]),
 tensor([[[4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465]],
 
         [[4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.

**Gating** 

Now we're ready to add sigmoid based gating on top of our attention. This gating uses the Hadamard product of the gate and the attention, which lets the model learn to suppress or pass through attention output per dimension. This allows the model to decide how much of the attention result to actually use for each dimension. We use a learned linear weight and sigmoid to pull gating values between 0 and 1, allowing the model to turn down specific values. 

In [59]:
gate = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1*(rows + cols)  

gate.weight = nn.Parameter(pattern)
gate.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000]], requires_grad=True)

In [60]:
y = torch.sigmoid(gate(x_norm)) * y
y.shape, y

(torch.Size([2, 9, 6]),
 tensor([[[1.6656, 1.0135, 0.4362, 0.4505, 0.1963, 0.1514],
          [1.6656, 1.0135, 0.4362, 0.4505, 0.1963, 0.1514],
          [2.8952, 3.2934, 3.4283, 3.9711, 4.0399, 4.1624],
          [2.8952, 3.2934, 3.4283, 3.9711, 4.0399, 4.1624],
          [1.6656, 1.0135, 0.4362, 0.4505, 0.1963, 0.1514],
          [1.6656, 1.0135, 0.4362, 0.4505, 0.1963, 0.1514],
          [2.8952, 3.2934, 3.4283, 3.9711, 4.0399, 4.1624],
          [2.8952, 3.2934, 3.4283, 3.9711, 4.0399, 4.1624],
          [1.6656, 1.0135, 0.4362, 0.4505, 0.1963, 0.1514]],
 
         [[1.8888, 1.1493, 0.4947, 0.5108, 0.2226, 0.1717],
          [3.2832, 3.7347, 3.8876, 4.5031, 4.5811, 4.7201],
          [1.8888, 1.1493, 0.4947, 0.5108, 0.2226, 0.1717],
          [1.8888, 1.1493, 0.4947, 0.5108, 0.2226, 0.1717],
          [3.2832, 3.7347, 3.8876, 4.5031, 4.5811, 4.7201],
          [3.2832, 3.7347, 3.8876, 4.5031, 4.5811, 4.7201],
          [1.8888, 1.1493, 0.4947, 0.5108, 0.2226, 0.1717],
          [3.

**Cross-head final projection** 

Finally, we will now project the gated attention matrix on another final linear layer. This allows the model to learn how to combine information across the different heads. 

In [61]:
c_proj = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 0.01*cols)  

c_proj.weight = nn.Parameter(pattern)
c_proj.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.1100, 0.1100, 0.1100, 0.1100, 0.1100, 0.1100],
        [0.1200, 0.1200, 0.1200, 0.1200, 0.1200, 0.1200],
        [0.1300, 0.1300, 0.1300, 0.1300, 0.1300, 0.1300],
        [0.1400, 0.1400, 0.1400, 0.1400, 0.1400, 0.1400],
        [0.1500, 0.1500, 0.1500, 0.1500, 0.1500, 0.1500]], requires_grad=True)

In [62]:
x_attn = c_proj(y)
x_attn.shape, x_attn

(torch.Size([2, 9, 6]),
 tensor([[[0.4214, 0.4117, 0.6743, 0.8992, 0.3045, 0.2333],
          [0.4214, 0.4117, 0.6743, 0.8992, 0.3045, 0.2333],
          [2.2091, 2.3782, 2.8195, 3.2232, 2.8072, 2.9148],
          [2.2091, 2.3782, 2.8195, 3.2232, 2.8072, 2.9148],
          [0.4214, 0.4117, 0.6743, 0.8992, 0.3045, 0.2333],
          [0.4214, 0.4117, 0.6743, 0.8992, 0.3045, 0.2333],
          [2.2091, 2.3782, 2.8195, 3.2232, 2.8072, 2.9148],
          [2.2091, 2.3782, 2.8195, 3.2232, 2.8072, 2.9148],
          [0.4214, 0.4117, 0.6743, 0.8992, 0.3045, 0.2333]],
 
         [[0.4738, 0.4694, 0.7372, 0.9674, 0.3779, 0.3119],
          [2.5010, 2.6993, 3.1698, 3.6027, 3.2160, 3.3527],
          [0.4738, 0.4694, 0.7372, 0.9674, 0.3779, 0.3119],
          [0.4738, 0.4694, 0.7372, 0.9674, 0.3779, 0.3119],
          [2.5010, 2.6993, 3.1699, 3.6027, 3.2160, 3.3527],
          [2.5010, 2.6993, 3.1699, 3.6027, 3.2160, 3.3527],
          [0.4738, 0.4694, 0.7372, 0.9674, 0.3779, 0.3119],
          [2.

#### Residual Connection

We now have our gated linear attention calculated and will use a residual connection to allow gradients to bypass the attention matrix. With this you'll see how much larger the residual connection impact is on our output compared to our attention. 

In [63]:
x = x + x_attn
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-10.5198, -10.5295, -10.2669, -10.0420, -10.6367, -10.7079],
          [ -4.5198,  -4.5295,  -4.2669,  -4.0420,  -4.6367,  -4.7079],
          [  8.5141,   8.6832,   9.1245,   9.5282,   9.1123,   9.2198],
          [  3.3242,   3.4933,   3.9346,   4.3383,   3.9223,   4.0299],
          [-10.5198, -10.5295, -10.2669, -10.0420, -10.6367, -10.7079],
          [-10.5198, -10.5295, -10.2669, -10.0420, -10.6367, -10.7079],
          [ 12.2022,  12.3713,  12.8127,  13.2163,  12.8004,  12.9079],
          [  6.6601,   6.8292,   7.2705,   7.6742,   7.2583,   7.3658],
          [ -4.5198,  -4.5295,  -4.2669,  -4.0420,  -4.6367,  -4.7079]],
 
         [[-10.4473, -10.4517, -10.1839,  -9.9538, -10.5433, -10.6092],
          [  2.8071,   3.0054,   3.4759,   3.9088,   3.5220,   3.6588],
          [ -4.4473,  -4.4517,  -4.1839,  -3.9538,  -4.5433,  -4.6092],
          [-10.4473, -10.4517, -10.1839,  -9.9538, -10.5433, -10.6092],
          [  5.0993,   5.2976,   5.76

#### RMSNorm 2
Now that we've calculated the attention, we'll do another round of normalization. We'll use RMSNorm again. 

In [64]:
rms2 = RMSNorm(embed_dim)
rms2.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [65]:
x_norm2 = rms2(x)
x_norm2.shape, x_norm2

(torch.Size([2, 9, 6]),
 tensor([[[-1.0064, -1.0073, -0.9822, -0.9607, -1.0176, -1.0244],
          [-1.0142, -1.0164, -0.9575, -0.9070, -1.0405, -1.0565],
          [ 0.9422,  0.9609,  1.0097,  1.0544,  1.0084,  1.0203],
          [ 0.8622,  0.9061,  1.0206,  1.1253,  1.0174,  1.0453],
          [-1.0064, -1.0073, -0.9822, -0.9607, -1.0176, -1.0244],
          [-1.0064, -1.0073, -0.9822, -0.9607, -1.0176, -1.0244],
          [ 0.9591,  0.9724,  1.0070,  1.0388,  1.0061,  1.0145],
          [ 0.9270,  0.9506,  1.0120,  1.0682,  1.0103,  1.0253],
          [-1.0142, -1.0164, -0.9575, -0.9070, -1.0405, -1.0565]],
 
         [[-1.0077, -1.0081, -0.9823, -0.9601, -1.0170, -1.0233],
          [ 0.8215,  0.8795,  1.0172,  1.1439,  1.0307,  1.0707],
          [-1.0175, -1.0185, -0.9573, -0.9046, -1.0395, -1.0546],
          [-1.0077, -1.0081, -0.9823, -0.9601, -1.0170, -1.0233],
          [ 0.8945,  0.9292,  1.0118,  1.0877,  1.0199,  1.0438],
          [ 0.8810,  0.9201,  1.0129,  1.0983,  1

#### SwiGLU 

Next, we'll introduce our scaling nonlinearity layers. Traditionally this was an MLP, but we replaced it with a swish-gated linear unit, or SwiGLU. SwiGLU replaces the single linear transform in a standard MLP with a gated pathway with a result as follows:

$$\begin{aligned}
\text{SiLU}(Z) &= Z \odot \sigma(Z) \\
\text{gate} &= \text{SiLU}(x W_g^\top) \\
H &= x W_u^\top \\
y &= (\text{gate} \odot H) W_d^\top
\end{aligned}$$

The Hadamard product based gating lets the network learn to selectively amplify or suppress features before the final projection, giving it more expressive power per parameter than a standard two-layer MLP with ReLU/GELU at the cost of some extra compute.

*You'll notice that SwiGLU has 3 weight matrices $W_g, W_u, W_d$, instead of the typical 2 we use in MLP. In production code, you might see helper functions that convert MLP ratios to SwiGLU ratios using 2/3 multiples to maintain the number of parameters.*

In [66]:
hidden_dim = 8

**Gate** 

We'll start by initializing our gating weight $W_g$ and calculating our gate. The gate will need to scale up to our hidden dimension as it will multiply directly against our weighted input. 

In [67]:
wg = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wg.weight, 0.5)
wg.weight

Parameter containing:
tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000]], requires_grad=True)

In [68]:
xwg = wg(x_norm2)
xwg.shape, xwg

(torch.Size([2, 9, 8]),
 tensor([[[-2.9993, -2.9993, -2.9993, -2.9993, -2.9993, -2.9993, -2.9993,
           -2.9993],
          [-2.9961, -2.9961, -2.9961, -2.9961, -2.9961, -2.9961, -2.9961,
           -2.9961],
          [ 2.9979,  2.9979,  2.9979,  2.9979,  2.9979,  2.9979,  2.9979,
            2.9979],
          [ 2.9884,  2.9884,  2.9884,  2.9884,  2.9884,  2.9884,  2.9884,
            2.9884],
          [-2.9993, -2.9993, -2.9993, -2.9993, -2.9993, -2.9993, -2.9993,
           -2.9993],
          [-2.9993, -2.9993, -2.9993, -2.9993, -2.9993, -2.9993, -2.9993,
           -2.9993],
          [ 2.9989,  2.9989,  2.9989,  2.9989,  2.9989,  2.9989,  2.9989,
            2.9989],
          [ 2.9967,  2.9967,  2.9967,  2.9967,  2.9967,  2.9967,  2.9967,
            2.9967],
          [-2.9961, -2.9961, -2.9961, -2.9961, -2.9961, -2.9961, -2.9961,
           -2.9961]],
 
         [[-2.9993, -2.9993, -2.9993, -2.9993, -2.9993, -2.9993, -2.9993,
           -2.9993],
          [ 2.9817,  2.

**SiLU Non-Linearity**

SiLU will pull our negative values closer to zero. Values above 1 will remain almost linear. When combined with the learned weights, you can quickly see how this becomes a gate. 

In [69]:
xwg = F.silu(xwg)
xwg.shape, xwg

(torch.Size([2, 9, 8]),
 tensor([[[-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [-0.1426, -0.1426, -0.1426, -0.1426, -0.1426, -0.1426, -0.1426,
           -0.1426],
          [ 2.8554,  2.8554,  2.8554,  2.8554,  2.8554,  2.8554,  2.8554,
            2.8554],
          [ 2.8451,  2.8451,  2.8451,  2.8451,  2.8451,  2.8451,  2.8451,
            2.8451],
          [-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [ 2.8566,  2.8566,  2.8566,  2.8566,  2.8566,  2.8566,  2.8566,
            2.8566],
          [ 2.8541,  2.8541,  2.8541,  2.8541,  2.8541,  2.8541,  2.8541,
            2.8541],
          [-0.1426, -0.1426, -0.1426, -0.1426, -0.1426, -0.1426, -0.1426,
           -0.1426]],
 
         [[-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [ 2.8378,  2.

**Weighted Input** 

Now we'll need to scale up our input to the hidden dimension. We'll use a weighted layer $W_u$ allowing the model to determine how to use the different channels to create the new dimensions.

In [70]:
wu = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wu.weight, -0.1)
wu.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [71]:
xwu = wu(x_norm2)
xwu.shape, xwu

(torch.Size([2, 9, 8]),
 tensor([[[ 0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,
            0.5999],
          [ 0.5992,  0.5992,  0.5992,  0.5992,  0.5992,  0.5992,  0.5992,
            0.5992],
          [-0.5996, -0.5996, -0.5996, -0.5996, -0.5996, -0.5996, -0.5996,
           -0.5996],
          [-0.5977, -0.5977, -0.5977, -0.5977, -0.5977, -0.5977, -0.5977,
           -0.5977],
          [ 0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,
            0.5999],
          [ 0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,
            0.5999],
          [-0.5998, -0.5998, -0.5998, -0.5998, -0.5998, -0.5998, -0.5998,
           -0.5998],
          [-0.5993, -0.5993, -0.5993, -0.5993, -0.5993, -0.5993, -0.5993,
           -0.5993],
          [ 0.5992,  0.5992,  0.5992,  0.5992,  0.5992,  0.5992,  0.5992,
            0.5992]],
 
         [[ 0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,  0.5999,
            0.5999],
          [-0.5963, -0.

**Apply Gate** Now we'll go ahead and apply the gate. We take the Hadamard product which allows the model to gate each value of the scaled up projection. 

In [72]:
xw = xwg * xwu
xw.shape, xw

(torch.Size([2, 9, 8]),
 tensor([[[-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-0.0855, -0.0855, -0.0855, -0.0855, -0.0855, -0.0855, -0.0855,
           -0.0855],
          [-1.7121, -1.7121, -1.7121, -1.7121, -1.7121, -1.7121, -1.7121,
           -1.7121],
          [-1.7005, -1.7005, -1.7005, -1.7005, -1.7005, -1.7005, -1.7005,
           -1.7005],
          [-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-1.7133, -1.7133, -1.7133, -1.7133, -1.7133, -1.7133, -1.7133,
           -1.7133],
          [-1.7106, -1.7106, -1.7106, -1.7106, -1.7106, -1.7106, -1.7106,
           -1.7106],
          [-0.0855, -0.0855, -0.0855, -0.0855, -0.0855, -0.0855, -0.0855,
           -0.0855]],
 
         [[-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-1.6923, -1.

**Project Down** 

Now we need to project back down to our embedding dimension. We'll use a final weighted $W_d$ layer to determine how to project back down. This is similar to the final layer of an MLP. 

In [73]:
wd = nn.Linear(hidden_dim, embed_dim, bias=False)
nn.init.constant_(wd.weight, 0.33)
wd.weight

Parameter containing:
tensor([[0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300]],
       requires_grad=True)

In [74]:
xswig = wd(xw)
xswig.shape, xswig

(torch.Size([2, 9, 6]),
 tensor([[[-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-0.2256, -0.2256, -0.2256, -0.2256, -0.2256, -0.2256],
          [-4.5198, -4.5198, -4.5198, -4.5198, -4.5198, -4.5198],
          [-4.4893, -4.4893, -4.4893, -4.4893, -4.4893, -4.4893],
          [-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-4.5232, -4.5232, -4.5232, -4.5232, -4.5232, -4.5232],
          [-4.5159, -4.5159, -4.5159, -4.5159, -4.5159, -4.5159],
          [-0.2256, -0.2256, -0.2256, -0.2256, -0.2256, -0.2256]],
 
         [[-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-4.4676, -4.4676, -4.4676, -4.4676, -4.4676, -4.4676],
          [-0.2256, -0.2256, -0.2256, -0.2256, -0.2256, -0.2256],
          [-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-4.5054, -4.5054, -4.5054, -4.5054, -4.5054, -4.5054],
          [-4.4998, -4.4998, -4.4998, -4.4998, -4

#### Residual Connection 2

We now have our SwiGLU-based projection calculated and will use a residual connection to allow gradients to bypass the SwiGLU calculations. With this you'll see how much larger the residual connection impact is on our output compared to our SwiGLU output. 

In [75]:
x = x + xswig
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-10.7452, -10.7549, -10.4923, -10.2674, -10.8622, -10.9334],
          [ -4.7454,  -4.7551,  -4.4926,  -4.2676,  -4.8624,  -4.9336],
          [  3.9943,   4.1634,   4.6047,   5.0084,   4.5924,   4.7000],
          [ -1.1651,  -0.9960,  -0.5547,  -0.1510,  -0.5670,  -0.4594],
          [-10.7452, -10.7549, -10.4923, -10.2674, -10.8622, -10.9334],
          [-10.7452, -10.7549, -10.4923, -10.2674, -10.8622, -10.9334],
          [  7.6790,   7.8481,   8.2894,   8.6931,   8.2772,   8.3847],
          [  2.1442,   2.3133,   2.7546,   3.1583,   2.7424,   2.8499],
          [ -4.7454,  -4.7551,  -4.4926,  -4.2676,  -4.8624,  -4.9336]],
 
         [[-10.6727, -10.6772, -10.4094, -10.1792, -10.7687, -10.8346],
          [ -1.6605,  -1.4622,  -0.9917,  -0.5588,  -0.9456,  -0.8088],
          [ -4.6729,  -4.6774,  -4.4096,  -4.1794,  -4.7689,  -4.8349],
          [-10.6727, -10.6772, -10.4094, -10.1792, -10.7687, -10.8346],
          [  0.5939,   0.7922,   1.26

### Final Layer Normalization
The previous layers can run sequentially for as many layers as are configured. The more layers, the "deeper" the network becomes. Once all the layers have completed, we're ready for a final normalization. Like previous normalizations this will pull our values together to focus on the variance. This output then becomes the latent representation of the context or target.

In [76]:
rmsf = RMSNorm(embed_dim)
rmsf.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [77]:
x = rmsf(x)
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [-1.0136, -1.0157, -0.9596, -0.9116, -1.0386, -1.0538],
          [ 0.8831,  0.9205,  1.0180,  1.1073,  1.0153,  1.0391],
          [-1.5923, -1.3612, -0.7581, -0.2064, -0.7748, -0.6279],
          [-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [ 0.9362,  0.9568,  1.0106,  1.0598,  1.0091,  1.0222],
          [ 0.7995,  0.8626,  1.0271,  1.1777,  1.0226,  1.0627],
          [-1.0136, -1.0157, -0.9596, -0.9116, -1.0386, -1.0538]],
 
         [[-1.0076, -1.0080, -0.9827, -0.9610, -1.0166, -1.0228],
          [-1.4621, -1.2875, -0.8732, -0.4920, -0.8326, -0.7122],
          [-1.0167, -1.0177, -0.9594, -0.9093, -1.0376, -1.0520],
          [-1.0076, -1.0080, -0.9827, -0.9610, -1.0166, -1.0228],
          [ 0.4783,  0.6379,  1.0168,  1.3654,  1.0540,  1.1641],
          [-0.0514,  0.2442,  0.9456,  1.5908,  1

### Output Latent
Now we've fully processed our input to create the context latent. This can now either be used for downstream models, or passed into the masked predictor during our encoder training.

In [78]:
context_latent = x
context_latent.shape, context_latent

(torch.Size([2, 9, 6]),
 tensor([[[-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [-1.0136, -1.0157, -0.9596, -0.9116, -1.0386, -1.0538],
          [ 0.8831,  0.9205,  1.0180,  1.1073,  1.0153,  1.0391],
          [-1.5923, -1.3612, -0.7581, -0.2064, -0.7748, -0.6279],
          [-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [ 0.9362,  0.9568,  1.0106,  1.0598,  1.0091,  1.0222],
          [ 0.7995,  0.8626,  1.0271,  1.1777,  1.0226,  1.0627],
          [-1.0136, -1.0157, -0.9596, -0.9116, -1.0386, -1.0538]],
 
         [[-1.0076, -1.0080, -0.9827, -0.9610, -1.0166, -1.0228],
          [-1.4621, -1.2875, -0.8732, -0.4920, -0.8326, -0.7122],
          [-1.0167, -1.0177, -0.9594, -0.9093, -1.0376, -1.0520],
          [-1.0076, -1.0080, -0.9827, -0.9610, -1.0166, -1.0228],
          [ 0.4783,  0.6379,  1.0168,  1.3654,  1.0540,  1.1641],
          [-0.0514,  0.2442,  0.9456,  1.5908,  1

## Masked Predictor
During encoder training, when we have a masked context input, we process the encoder's full output with a masked predictor. The masked predictor returns a predicted latent for every gene position, keeping the same `[batch, genes, embedding]` shape as the encoder output. We later use the masks when choosing which positions to score during reconstruction.

This keeps the reconstruction objective separate from the encoder's cell representation. The predictor uses the same transformer block as the encoder, followed by a final linear head, so we'll only call out the differences here.

### Transformer Block 
The masked predictor uses the same transformer block consisting of:

1. RMS normalization
2. Gated linear self-attention
3. RMS normalization
4. SwiGLU

Residual connections provide gradient bypassing around the attention and SwiGLU layers. This set of 4 units is repeated based on how many layers are configured. 

#### RMSNorm M1

In [79]:
rms_m1 = RMSNorm(embed_dim)

rms_m1.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [80]:
x_norm = rms_m1(x)
x_norm.shape, x_norm

(torch.Size([2, 9, 6]),
 tensor([[[-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [-1.0136, -1.0157, -0.9596, -0.9116, -1.0386, -1.0538],
          [ 0.8831,  0.9205,  1.0180,  1.1073,  1.0153,  1.0391],
          [-1.5923, -1.3612, -0.7581, -0.2064, -0.7748, -0.6279],
          [-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [ 0.9362,  0.9568,  1.0106,  1.0598,  1.0091,  1.0222],
          [ 0.7995,  0.8626,  1.0271,  1.1777,  1.0226,  1.0627],
          [-1.0136, -1.0157, -0.9596, -0.9116, -1.0386, -1.0538]],
 
         [[-1.0076, -1.0080, -0.9827, -0.9610, -1.0166, -1.0228],
          [-1.4621, -1.2875, -0.8732, -0.4920, -0.8326, -0.7122],
          [-1.0167, -1.0177, -0.9594, -0.9093, -1.0376, -1.0520],
          [-1.0076, -1.0080, -0.9827, -0.9610, -1.0166, -1.0228],
          [ 0.4783,  0.6379,  1.0168,  1.3654,  1.0540,  1.1641],
          [-0.0514,  0.2442,  0.9456,  1.5908,  1

#### Multi-Headed Gated Linear Attention

In [81]:
B, T_q, C = x_norm.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

(2, 9, 6, 3)

**Self-attention**

In [82]:
kv_input = x_norm
T_kv = kv_input.size(1)
kv_input, T_kv

(tensor([[[-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [-1.0136, -1.0157, -0.9596, -0.9116, -1.0386, -1.0538],
          [ 0.8831,  0.9205,  1.0180,  1.1073,  1.0153,  1.0391],
          [-1.5923, -1.3612, -0.7581, -0.2064, -0.7748, -0.6279],
          [-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
          [ 0.9362,  0.9568,  1.0106,  1.0598,  1.0091,  1.0222],
          [ 0.7995,  0.8626,  1.0271,  1.1777,  1.0226,  1.0627],
          [-1.0136, -1.0157, -0.9596, -0.9116, -1.0386, -1.0538]],
 
         [[-1.0076, -1.0080, -0.9827, -0.9610, -1.0166, -1.0228],
          [-1.4621, -1.2875, -0.8732, -0.4920, -0.8326, -0.7122],
          [-1.0167, -1.0177, -0.9594, -0.9093, -1.0376, -1.0520],
          [-1.0076, -1.0080, -0.9827, -0.9610, -1.0166, -1.0228],
          [ 0.4783,  0.6379,  1.0168,  1.3654,  1.0540,  1.1641],
          [-0.0514,  0.2442,  0.9456,  1.5908,  1.0143,  1.2182],
       

**Query**

In [83]:
q_proj_m = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(q_proj_m.weight, -0.1)
nn.init.constant_(q_proj_m.bias, 0)
q_proj_m.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [84]:
q = q_proj_m(x_norm).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

(torch.Size([2, 2, 9, 3]),
 tensor([[[[ 0.5999,  0.5999,  0.5999],
           [ 0.5993,  0.5993,  0.5993],
           [-0.5983, -0.5983, -0.5983],
           [ 0.5321,  0.5321,  0.5321],
           [ 0.5999,  0.5999,  0.5999],
           [ 0.5999,  0.5999,  0.5999],
           [-0.5995, -0.5995, -0.5995],
           [-0.5952, -0.5952, -0.5952],
           [ 0.5993,  0.5993,  0.5993]],
 
          [[ 0.5999,  0.5999,  0.5999],
           [ 0.5993,  0.5993,  0.5993],
           [-0.5983, -0.5983, -0.5983],
           [ 0.5321,  0.5321,  0.5321],
           [ 0.5999,  0.5999,  0.5999],
           [ 0.5999,  0.5999,  0.5999],
           [-0.5995, -0.5995, -0.5995],
           [-0.5952, -0.5952, -0.5952],
           [ 0.5993,  0.5993,  0.5993]]],
 
 
         [[[ 0.5999,  0.5999,  0.5999],
           [ 0.5659,  0.5659,  0.5659],
           [ 0.5993,  0.5993,  0.5993],
           [ 0.5999,  0.5999,  0.5999],
           [-0.5717, -0.5717, -0.5717],
           [-0.4962, -0.4962, -0.4962],
    

In [85]:
q = F.elu(q) + 1.0
q.shape, q

(torch.Size([2, 2, 9, 3]),
 tensor([[[[1.5999, 1.5999, 1.5999],
           [1.5993, 1.5993, 1.5993],
           [0.5497, 0.5497, 0.5497],
           [1.5321, 1.5321, 1.5321],
           [1.5999, 1.5999, 1.5999],
           [1.5999, 1.5999, 1.5999],
           [0.5491, 0.5491, 0.5491],
           [0.5514, 0.5514, 0.5514],
           [1.5993, 1.5993, 1.5993]],
 
          [[1.5999, 1.5999, 1.5999],
           [1.5993, 1.5993, 1.5993],
           [0.5497, 0.5497, 0.5497],
           [1.5321, 1.5321, 1.5321],
           [1.5999, 1.5999, 1.5999],
           [1.5999, 1.5999, 1.5999],
           [0.5491, 0.5491, 0.5491],
           [0.5514, 0.5514, 0.5514],
           [1.5993, 1.5993, 1.5993]]],
 
 
         [[[1.5999, 1.5999, 1.5999],
           [1.5659, 1.5659, 1.5659],
           [1.5993, 1.5993, 1.5993],
           [1.5999, 1.5999, 1.5999],
           [0.5646, 0.5646, 0.5646],
           [0.6089, 0.6089, 0.6089],
           [1.5999, 1.5999, 1.5999],
           [0.5489, 0.5489, 0.5489],
  

**Key**

In [86]:
k_proj_m = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(k_proj_m.weight, 0.2)
nn.init.constant_(k_proj_m.bias, 0)
k_proj_m.weight

Parameter containing:
tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000]], requires_grad=True)

In [87]:
k = k_proj_m(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

(torch.Size([2, 2, 9, 3]),
 tensor([[[[-1.1997, -1.1997, -1.1997],
           [-1.1986, -1.1986, -1.1986],
           [ 1.1966,  1.1966,  1.1966],
           [-1.0641, -1.0641, -1.0641],
           [-1.1997, -1.1997, -1.1997],
           [-1.1997, -1.1997, -1.1997],
           [ 1.1990,  1.1990,  1.1990],
           [ 1.1904,  1.1904,  1.1904],
           [-1.1986, -1.1986, -1.1986]],
 
          [[-1.1997, -1.1997, -1.1997],
           [-1.1986, -1.1986, -1.1986],
           [ 1.1966,  1.1966,  1.1966],
           [-1.0641, -1.0641, -1.0641],
           [-1.1997, -1.1997, -1.1997],
           [-1.1997, -1.1997, -1.1997],
           [ 1.1990,  1.1990,  1.1990],
           [ 1.1904,  1.1904,  1.1904],
           [-1.1986, -1.1986, -1.1986]]],
 
 
         [[[-1.1997, -1.1997, -1.1997],
           [-1.1319, -1.1319, -1.1319],
           [-1.1985, -1.1985, -1.1985],
           [-1.1997, -1.1997, -1.1997],
           [ 1.1433,  1.1433,  1.1433],
           [ 0.9924,  0.9924,  0.9924],
    

In [88]:
k = F.elu(k) + 1.0
k

tensor([[[[0.3013, 0.3013, 0.3013],
          [0.3016, 0.3016, 0.3016],
          [2.1966, 2.1966, 2.1966],
          [0.3450, 0.3450, 0.3450],
          [0.3013, 0.3013, 0.3013],
          [0.3013, 0.3013, 0.3013],
          [2.1990, 2.1990, 2.1990],
          [2.1904, 2.1904, 2.1904],
          [0.3016, 0.3016, 0.3016]],

         [[0.3013, 0.3013, 0.3013],
          [0.3016, 0.3016, 0.3016],
          [2.1966, 2.1966, 2.1966],
          [0.3450, 0.3450, 0.3450],
          [0.3013, 0.3013, 0.3013],
          [0.3013, 0.3013, 0.3013],
          [2.1990, 2.1990, 2.1990],
          [2.1904, 2.1904, 2.1904],
          [0.3016, 0.3016, 0.3016]]],


        [[[0.3013, 0.3013, 0.3013],
          [0.3224, 0.3224, 0.3224],
          [0.3016, 0.3016, 0.3016],
          [0.3013, 0.3013, 0.3013],
          [2.1433, 2.1433, 2.1433],
          [1.9924, 1.9924, 1.9924],
          [0.3013, 0.3013, 0.3013],
          [2.1997, 2.1997, 2.1997],
          [2.1992, 2.1992, 2.1992]],

         [[0.3013, 0

**Value**

In [89]:
v_proj_m = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(v_proj_m.weight, 1.0)
nn.init.constant_(v_proj_m.bias, 0)
v_proj_m.weight

Parameter containing:
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]], requires_grad=True)

In [90]:
v = v_proj_m(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

(torch.Size([2, 2, 9, 3]),
 tensor([[[[-5.9986, -5.9986, -5.9986],
           [-5.9929, -5.9929, -5.9929],
           [ 5.9832,  5.9832,  5.9832],
           [-5.3206, -5.3206, -5.3206],
           [-5.9986, -5.9986, -5.9986],
           [-5.9986, -5.9986, -5.9986],
           [ 5.9949,  5.9949,  5.9949],
           [ 5.9521,  5.9521,  5.9521],
           [-5.9929, -5.9929, -5.9929]],
 
          [[-5.9986, -5.9986, -5.9986],
           [-5.9929, -5.9929, -5.9929],
           [ 5.9832,  5.9832,  5.9832],
           [-5.3206, -5.3206, -5.3206],
           [-5.9986, -5.9986, -5.9986],
           [-5.9986, -5.9986, -5.9986],
           [ 5.9949,  5.9949,  5.9949],
           [ 5.9521,  5.9521,  5.9521],
           [-5.9929, -5.9929, -5.9929]]],
 
 
         [[[-5.9986, -5.9986, -5.9986],
           [-5.6594, -5.6594, -5.6594],
           [-5.9927, -5.9927, -5.9927],
           [-5.9986, -5.9986, -5.9986],
           [ 5.7165,  5.7165,  5.7165],
           [ 4.9618,  4.9618,  4.9618],
    

**Normalize Denominator**

In [91]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

(torch.Size([2, 2, 3, 1]),
 tensor([[[[ 8.4381],
           [ 8.4381],
           [ 8.4381]],
 
          [[ 8.4381],
           [ 8.4381],
           [ 8.4381]]],
 
 
         [[[10.0624],
           [10.0624],
           [10.0624]],
 
          [[10.0624],
           [10.0624],
           [10.0624]]]], grad_fn=<UnsqueezeBackward0>))

**Denominator**

In [92]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

(torch.Size([2, 2, 9, 1]),
 tensor([[[[0.0247],
           [0.0247],
           [0.0719],
           [0.0258],
           [0.0247],
           [0.0247],
           [0.0719],
           [0.0716],
           [0.0247]],
 
          [[0.0247],
           [0.0247],
           [0.0719],
           [0.0258],
           [0.0247],
           [0.0247],
           [0.0719],
           [0.0716],
           [0.0247]]],
 
 
         [[[0.0207],
           [0.0212],
           [0.0207],
           [0.0207],
           [0.0587],
           [0.0544],
           [0.0207],
           [0.0604],
           [0.0603]],
 
          [[0.0207],
           [0.0212],
           [0.0207],
           [0.0207],
           [0.0587],
           [0.0544],
           [0.0207],
           [0.0604],
           [0.0603]]]], grad_fn=<MulBackward0>))

**Numerator**

In [93]:
kv = k.transpose(-2, -1) @ v
kv.shape, kv

(torch.Size([2, 2, 3, 3]),
 tensor([[[[28.4904, 28.4904, 28.4904],
           [28.4904, 28.4904, 28.4904],
           [28.4904, 28.4904, 28.4904]],
 
          [[28.4904, 28.4904, 28.4904],
           [28.4904, 28.4904, 28.4904],
           [28.4904, 28.4904, 28.4904]]],
 
 
         [[[39.4648, 39.4648, 39.4648],
           [39.4648, 39.4648, 39.4648],
           [39.4648, 39.4648, 39.4648]],
 
          [[39.4648, 39.4648, 39.4648],
           [39.4648, 39.4648, 39.4648],
           [39.4648, 39.4648, 39.4648]]]], grad_fn=<UnsafeViewBackward0>))

In [94]:
qkv = q @ kv
qkv.shape, qkv

(torch.Size([2, 2, 9, 3]),
 tensor([[[[136.7424, 136.7424, 136.7424],
           [136.6931, 136.6931, 136.6931],
           [ 46.9865,  46.9865,  46.9865],
           [130.9473, 130.9473, 130.9473],
           [136.7424, 136.7424, 136.7424],
           [136.7424, 136.7424, 136.7424],
           [ 46.9316,  46.9316,  46.9316],
           [ 47.1329,  47.1329,  47.1329],
           [136.6931, 136.6931, 136.6931]],
 
          [[136.7424, 136.7424, 136.7424],
           [136.6931, 136.6931, 136.6931],
           [ 46.9865,  46.9865,  46.9865],
           [130.9473, 130.9473, 130.9473],
           [136.7424, 136.7424, 136.7424],
           [136.7424, 136.7424, 136.7424],
           [ 46.9316,  46.9316,  46.9316],
           [ 47.1329,  47.1329,  47.1329],
           [136.6931, 136.6931, 136.6931]]],
 
 
         [[[189.4148, 189.4148, 189.4148],
           [185.3988, 185.3988, 185.3988],
           [189.3447, 189.3447, 189.3447],
           [189.4148, 189.4148, 189.4148],
           [ 66.84

**Linear Attention**

In [95]:
y = qkv * z
y.shape, y

(torch.Size([2, 2, 9, 3]),
 tensor([[[[3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764]],
 
          [[3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764],
           [3.3764, 3.3764, 3.3764]]],
 
 
         [[[3.9220, 3.9220, 3.9220],
           [3.9220, 3.9220, 3.9220],
           [3.9220, 3.9220, 3.9220],
           [3.9220, 3.9220, 3.9220],
           [3.9220, 3.9220, 3.9220],
           [3.9220, 3.9220, 3.9220],
           [3.9220, 3.9220, 3.9220],
           [3.9220, 3.9220, 3.9220],
  

**Collapse Heads**

In [96]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

(torch.Size([2, 9, 6]),
 tensor([[[3.3764, 3.3764, 3.3764, 3.3764, 3.3764, 3.3764],
          [3.3764, 3.3764, 3.3764, 3.3764, 3.3764, 3.3764],
          [3.3764, 3.3764, 3.3764, 3.3764, 3.3764, 3.3764],
          [3.3764, 3.3764, 3.3764, 3.3764, 3.3764, 3.3764],
          [3.3764, 3.3764, 3.3764, 3.3764, 3.3764, 3.3764],
          [3.3764, 3.3764, 3.3764, 3.3764, 3.3764, 3.3764],
          [3.3764, 3.3764, 3.3764, 3.3764, 3.3764, 3.3764],
          [3.3764, 3.3764, 3.3764, 3.3764, 3.3764, 3.3764],
          [3.3764, 3.3764, 3.3764, 3.3764, 3.3764, 3.3764]],
 
         [[3.9220, 3.9220, 3.9220, 3.9220, 3.9220, 3.9220],
          [3.9220, 3.9220, 3.9220, 3.9220, 3.9220, 3.9220],
          [3.9220, 3.9220, 3.9220, 3.9220, 3.9220, 3.9220],
          [3.9220, 3.9220, 3.9220, 3.9220, 3.9220, 3.9220],
          [3.9220, 3.9220, 3.9220, 3.9220, 3.9220, 3.9220],
          [3.9220, 3.9220, 3.9220, 3.9220, 3.9220, 3.9220],
          [3.9220, 3.9220, 3.9220, 3.9220, 3.9220, 3.9220],
          [3.

**Gating**

In [97]:
gate_m = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1*(rows + cols)  

gate_m.weight = nn.Parameter(pattern)
gate_m.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000]], requires_grad=True)

In [98]:
y = torch.sigmoid(gate_m(x_norm)) * y
y.shape, y

(torch.Size([2, 9, 6]),
 tensor([[[1.3614, 0.7789, 0.3483, 0.3463, 0.1707, 0.1312],
          [1.3618, 0.7796, 0.3489, 0.3470, 0.1712, 0.1316],
          [2.3340, 2.5900, 2.7254, 3.1480, 3.2249, 3.3145],
          [1.4168, 0.8631, 0.4172, 0.4401, 0.2348, 0.1933],
          [1.3614, 0.7789, 0.3483, 0.3463, 0.1707, 0.1312],
          [1.3614, 0.7789, 0.3483, 0.3463, 0.1707, 0.1312],
          [2.3348, 2.5914, 2.7273, 3.1490, 3.2258, 3.3149],
          [2.3317, 2.5862, 2.7205, 3.1453, 3.2227, 3.3134],
          [1.3618, 0.7796, 0.3489, 0.3470, 0.1712, 0.1316]],
 
         [[1.5814, 0.9048, 0.4046, 0.4022, 0.1983, 0.1524],
          [1.6135, 0.9529, 0.4430, 0.4539, 0.2328, 0.1852],
          [1.5819, 0.9056, 0.4052, 0.4031, 0.1989, 0.1529],
          [1.5814, 0.9048, 0.4046, 0.4022, 0.1983, 0.1524],
          [2.6887, 2.9706, 3.1158, 3.6291, 3.7222, 3.8379],
          [2.6240, 2.8577, 2.9612, 3.5360, 3.6373, 3.7914],
          [1.5814, 0.9048, 0.4046, 0.4022, 0.1983, 0.1524],
          [2.

**Cross-head final projection**

In [99]:
c_proj_m = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 0.01*cols)  

c_proj_m.weight = nn.Parameter(pattern)
c_proj_m.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.1100, 0.1100, 0.1100, 0.1100, 0.1100, 0.1100],
        [0.1200, 0.1200, 0.1200, 0.1200, 0.1200, 0.1200],
        [0.1300, 0.1300, 0.1300, 0.1300, 0.1300, 0.1300],
        [0.1400, 0.1400, 0.1400, 0.1400, 0.1400, 0.1400],
        [0.1500, 0.1500, 0.1500, 0.1500, 0.1500, 0.1500]], requires_grad=True)

In [100]:
x_attn = c_proj_m(y)
x_attn.shape, x_attn

(torch.Size([2, 9, 6]),
 tensor([[[0.2380, 0.3836, 0.7540, 0.0133, 0.2176, 0.1681],
          [0.2383, 0.3840, 0.7544, 0.0138, 0.2181, 0.1686],
          [1.6580, 1.9456, 2.4580, 1.8594, 2.2057, 2.2981],
          [0.2808, 0.4307, 0.8054, 0.0690, 0.2776, 0.2324],
          [0.2380, 0.3836, 0.7540, 0.0133, 0.2176, 0.1681],
          [0.2380, 0.3836, 0.7540, 0.0133, 0.2176, 0.1681],
          [1.6586, 1.9463, 2.4588, 1.8602, 2.2065, 2.2991],
          [1.6563, 1.9437, 2.4560, 1.8571, 2.2033, 2.2956],
          [0.2383, 0.3840, 0.7544, 0.0138, 0.2181, 0.1686]],
 
         [[0.2887, 0.4394, 0.8148, 0.0792, 0.2886, 0.2442],
          [0.3124, 0.4655, 0.8433, 0.1101, 0.3219, 0.2798],
          [0.2891, 0.4398, 0.8153, 0.0797, 0.2892, 0.2447],
          [0.2887, 0.4394, 0.8148, 0.0792, 0.2886, 0.2442],
          [1.9208, 2.2346, 2.7733, 2.2009, 2.5735, 2.6923],
          [1.8651, 2.1734, 2.7065, 2.1285, 2.4956, 2.6087],
          [0.2887, 0.4394, 0.8148, 0.0792, 0.2886, 0.2442],
          [1.

#### Residual Connection

In [101]:
x = x + x_attn
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-0.7683, -0.6236, -0.2286, -0.9482, -0.7996, -0.8558],
          [-0.7753, -0.6317, -0.2052, -0.8978, -0.8205, -0.8852],
          [ 2.5411,  2.8661,  3.4760,  2.9666,  3.2210,  3.3372],
          [-1.3114, -0.9304,  0.0473, -0.1373, -0.4972, -0.3955],
          [-0.7683, -0.6236, -0.2286, -0.9482, -0.7996, -0.8558],
          [-0.7683, -0.6236, -0.2286, -0.9482, -0.7996, -0.8558],
          [ 2.5949,  2.9031,  3.4694,  2.9200,  3.2157,  3.3213],
          [ 2.4558,  2.8063,  3.4831,  3.0348,  3.2258,  3.3582],
          [-0.7753, -0.6317, -0.2052, -0.8978, -0.8205, -0.8852]],
 
         [[-0.7189, -0.5686, -0.1679, -0.8817, -0.7280, -0.7787],
          [-1.1496, -0.8220, -0.0299, -0.3819, -0.5107, -0.4324],
          [-0.7276, -0.5779, -0.1441, -0.8296, -0.7484, -0.8072],
          [-0.7189, -0.5686, -0.1679, -0.8817, -0.7280, -0.7787],
          [ 2.3990,  2.8726,  3.7901,  3.5664,  3.6275,  3.8564],
          [ 1.8137,  2.4176,  3.6520,  3.7194,  3

#### RMSNorm M2

In [102]:
rms2_m2 = RMSNorm(embed_dim)
rms2_m2.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [103]:
x_norm2 = rms2_m2(x)
x_norm2.shape, x_norm2

(torch.Size([2, 9, 6]),
 tensor([[[-1.0356, -0.8406, -0.3082, -1.2781, -1.0778, -1.1536],
          [-1.0446, -0.8512, -0.2765, -1.2097, -1.1055, -1.1927],
          [ 0.8240,  0.9293,  1.1271,  0.9619,  1.0444,  1.0821],
          [-1.8515, -1.3136,  0.0668, -0.1939, -0.7019, -0.5583],
          [-1.0356, -0.8406, -0.3082, -1.2781, -1.0778, -1.1536],
          [-1.0356, -0.8406, -0.3082, -1.2781, -1.0778, -1.1536],
          [ 0.8412,  0.9411,  1.1247,  0.9466,  1.0424,  1.0767],
          [ 0.7973,  0.9110,  1.1307,  0.9852,  1.0472,  1.0902],
          [-1.0446, -0.8512, -0.2765, -1.2097, -1.1055, -1.1927]],
 
         [[-1.0557, -0.8350, -0.2465, -1.2949, -1.0691, -1.1436],
          [-1.7492, -1.2507, -0.0454, -0.5811, -0.7770, -0.6579],
          [-1.0682, -0.8483, -0.2116, -1.2178, -1.0987, -1.1850],
          [-1.0557, -0.8350, -0.2465, -1.2949, -1.0691, -1.1436],
          [ 0.7068,  0.8463,  1.1167,  1.0507,  1.0687,  1.1362],
          [ 0.5585,  0.7445,  1.1246,  1.1454,  1

#### SwiGLU

**Gate**

In [104]:
wg_m = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wg_m.weight, 0.5)
wg_m.weight

Parameter containing:
tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000]], requires_grad=True)

In [105]:
xwg = wg_m(x_norm2)
xwg.shape, xwg

(torch.Size([2, 9, 8]),
 tensor([[[-2.8469, -2.8469, -2.8469, -2.8469, -2.8469, -2.8469, -2.8469,
           -2.8469],
          [-2.8401, -2.8401, -2.8401, -2.8401, -2.8401, -2.8401, -2.8401,
           -2.8401],
          [ 2.9844,  2.9844,  2.9844,  2.9844,  2.9844,  2.9844,  2.9844,
            2.9844],
          [-2.2762, -2.2762, -2.2762, -2.2762, -2.2762, -2.2762, -2.2762,
           -2.2762],
          [-2.8469, -2.8469, -2.8469, -2.8469, -2.8469, -2.8469, -2.8469,
           -2.8469],
          [-2.8469, -2.8469, -2.8469, -2.8469, -2.8469, -2.8469, -2.8469,
           -2.8469],
          [ 2.9863,  2.9863,  2.9863,  2.9863,  2.9863,  2.9863,  2.9863,
            2.9863],
          [ 2.9808,  2.9808,  2.9808,  2.9808,  2.9808,  2.9808,  2.9808,
            2.9808],
          [-2.8401, -2.8401, -2.8401, -2.8401, -2.8401, -2.8401, -2.8401,
           -2.8401]],
 
         [[-2.8224, -2.8224, -2.8224, -2.8224, -2.8224, -2.8224, -2.8224,
           -2.8224],
          [-2.5306, -2.

**SiLU Non-Linearity**

In [106]:
xwg = F.silu(xwg)
xwg.shape, xwg

(torch.Size([2, 9, 8]),
 tensor([[[-0.1561, -0.1561, -0.1561, -0.1561, -0.1561, -0.1561, -0.1561,
           -0.1561],
          [-0.1568, -0.1568, -0.1568, -0.1568, -0.1568, -0.1568, -0.1568,
           -0.1568],
          [ 2.8408,  2.8408,  2.8408,  2.8408,  2.8408,  2.8408,  2.8408,
            2.8408],
          [-0.2119, -0.2119, -0.2119, -0.2119, -0.2119, -0.2119, -0.2119,
           -0.2119],
          [-0.1561, -0.1561, -0.1561, -0.1561, -0.1561, -0.1561, -0.1561,
           -0.1561],
          [-0.1561, -0.1561, -0.1561, -0.1561, -0.1561, -0.1561, -0.1561,
           -0.1561],
          [ 2.8428,  2.8428,  2.8428,  2.8428,  2.8428,  2.8428,  2.8428,
            2.8428],
          [ 2.8369,  2.8369,  2.8369,  2.8369,  2.8369,  2.8369,  2.8369,
            2.8369],
          [-0.1568, -0.1568, -0.1568, -0.1568, -0.1568, -0.1568, -0.1568,
           -0.1568]],
 
         [[-0.1584, -0.1584, -0.1584, -0.1584, -0.1584, -0.1584, -0.1584,
           -0.1584],
          [-0.1866, -0.

**Weighted Input** 

In [107]:
wu_m = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wu_m.weight, -0.1)
wu_m.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [108]:
xwu = wu_m(x_norm2)
xwu.shape, xwu

(torch.Size([2, 9, 8]),
 tensor([[[ 0.5694,  0.5694,  0.5694,  0.5694,  0.5694,  0.5694,  0.5694,
            0.5694],
          [ 0.5680,  0.5680,  0.5680,  0.5680,  0.5680,  0.5680,  0.5680,
            0.5680],
          [-0.5969, -0.5969, -0.5969, -0.5969, -0.5969, -0.5969, -0.5969,
           -0.5969],
          [ 0.4552,  0.4552,  0.4552,  0.4552,  0.4552,  0.4552,  0.4552,
            0.4552],
          [ 0.5694,  0.5694,  0.5694,  0.5694,  0.5694,  0.5694,  0.5694,
            0.5694],
          [ 0.5694,  0.5694,  0.5694,  0.5694,  0.5694,  0.5694,  0.5694,
            0.5694],
          [-0.5973, -0.5973, -0.5973, -0.5973, -0.5973, -0.5973, -0.5973,
           -0.5973],
          [-0.5962, -0.5962, -0.5962, -0.5962, -0.5962, -0.5962, -0.5962,
           -0.5962],
          [ 0.5680,  0.5680,  0.5680,  0.5680,  0.5680,  0.5680,  0.5680,
            0.5680]],
 
         [[ 0.5645,  0.5645,  0.5645,  0.5645,  0.5645,  0.5645,  0.5645,
            0.5645],
          [ 0.5061,  0.

**Apply Gate**

In [109]:
xw = xwg * xwu
xw.shape, xw

(torch.Size([2, 9, 8]),
 tensor([[[-0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889,
           -0.0889],
          [-0.0890, -0.0890, -0.0890, -0.0890, -0.0890, -0.0890, -0.0890,
           -0.0890],
          [-1.6956, -1.6956, -1.6956, -1.6956, -1.6956, -1.6956, -1.6956,
           -1.6956],
          [-0.0965, -0.0965, -0.0965, -0.0965, -0.0965, -0.0965, -0.0965,
           -0.0965],
          [-0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889,
           -0.0889],
          [-0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889,
           -0.0889],
          [-1.6979, -1.6979, -1.6979, -1.6979, -1.6979, -1.6979, -1.6979,
           -1.6979],
          [-1.6912, -1.6912, -1.6912, -1.6912, -1.6912, -1.6912, -1.6912,
           -1.6912],
          [-0.0890, -0.0890, -0.0890, -0.0890, -0.0890, -0.0890, -0.0890,
           -0.0890]],
 
         [[-0.0894, -0.0894, -0.0894, -0.0894, -0.0894, -0.0894, -0.0894,
           -0.0894],
          [-0.0944, -0.

**Project Down** 

In [110]:
wd_m = nn.Linear(hidden_dim, embed_dim, bias=False)
nn.init.constant_(wd_m.weight, 0.33)
wd_m.weight

Parameter containing:
tensor([[0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300]],
       requires_grad=True)

In [111]:
xswig = wd_m(xw)
xswig.shape, xswig

(torch.Size([2, 9, 6]),
 tensor([[[-0.2347, -0.2347, -0.2347, -0.2347, -0.2347, -0.2347],
          [-0.2351, -0.2351, -0.2351, -0.2351, -0.2351, -0.2351],
          [-4.4764, -4.4764, -4.4764, -4.4764, -4.4764, -4.4764],
          [-0.2547, -0.2547, -0.2547, -0.2547, -0.2547, -0.2547],
          [-0.2347, -0.2347, -0.2347, -0.2347, -0.2347, -0.2347],
          [-0.2347, -0.2347, -0.2347, -0.2347, -0.2347, -0.2347],
          [-4.4825, -4.4825, -4.4825, -4.4825, -4.4825, -4.4825],
          [-4.4649, -4.4649, -4.4649, -4.4649, -4.4649, -4.4649],
          [-0.2351, -0.2351, -0.2351, -0.2351, -0.2351, -0.2351]],
 
         [[-0.2361, -0.2361, -0.2361, -0.2361, -0.2361, -0.2361],
          [-0.2493, -0.2493, -0.2493, -0.2493, -0.2493, -0.2493],
          [-0.2365, -0.2365, -0.2365, -0.2365, -0.2365, -0.2365],
          [-0.2361, -0.2361, -0.2361, -0.2361, -0.2361, -0.2361],
          [-4.4070, -4.4070, -4.4070, -4.4070, -4.4070, -4.4070],
          [-4.2596, -4.2596, -4.2596, -4.2596, -4

#### Residual Connection

In [112]:
x = x + xswig
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-1.0030, -0.8583, -0.4633, -1.1829, -1.0343, -1.0905],
          [-1.0104, -0.8668, -0.4403, -1.1329, -1.0556, -1.1203],
          [-1.9353, -1.6103, -1.0004, -1.5098, -1.2554, -1.1392],
          [-1.5661, -1.1852, -0.2074, -0.3921, -0.7519, -0.6502],
          [-1.0030, -0.8583, -0.4633, -1.1829, -1.0343, -1.0905],
          [-1.0030, -0.8583, -0.4633, -1.1829, -1.0343, -1.0905],
          [-1.8876, -1.5793, -1.0131, -1.5625, -1.2668, -1.1611],
          [-2.0090, -1.6585, -0.9818, -1.4301, -1.2390, -1.1066],
          [-1.0104, -0.8668, -0.4403, -1.1329, -1.0556, -1.1203]],
 
         [[-0.9549, -0.8047, -0.4039, -1.1178, -0.9641, -1.0148],
          [-1.3989, -1.0713, -0.2792, -0.6312, -0.7600, -0.6817],
          [-0.9641, -0.8144, -0.3806, -1.0661, -0.9849, -1.0437],
          [-0.9549, -0.8047, -0.4039, -1.1178, -0.9641, -1.0148],
          [-2.0079, -1.5344, -0.6168, -0.8406, -0.7795, -0.5506],
          [-2.4459, -1.8420, -0.6075, -0.5402, -0

### Post Attention Layer Normalization

In [113]:
rmsf_m = RMSNorm(embed_dim)
rmsf_m.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [114]:
x = rmsf_m(x)
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-1.0368, -0.8872, -0.4789, -1.2227, -1.0691, -1.1272],
          [-1.0441, -0.8957, -0.4550, -1.1707, -1.0908, -1.1577],
          [-1.3412, -1.1160, -0.6933, -1.0463, -0.8700, -0.7895],
          [-1.7084, -1.2928, -0.2262, -0.4277, -0.8202, -0.7092],
          [-1.0368, -0.8872, -0.4789, -1.2227, -1.0691, -1.1272],
          [-1.0368, -0.8872, -0.4789, -1.2227, -1.0691, -1.1272],
          [-1.3089, -1.0952, -0.7025, -1.0834, -0.8784, -0.8052],
          [-1.3888, -1.1465, -0.6787, -0.9886, -0.8565, -0.7650],
          [-1.0441, -0.8957, -0.4550, -1.1707, -1.0908, -1.1577]],
 
         [[-1.0533, -0.8876, -0.4456, -1.2330, -1.0634, -1.1193],
          [-1.5936, -1.2204, -0.3181, -0.7191, -0.8658, -0.7766],
          [-1.0632, -0.8981, -0.4198, -1.1757, -1.0862, -1.1510],
          [-1.0533, -0.8876, -0.4456, -1.2330, -1.0634, -1.1193],
          [-1.6986, -1.2980, -0.5218, -0.7111, -0.6594, -0.4658],
          [-1.8243, -1.3739, -0.4531, -0.4029, -0

### Masked Linear Head
Our masked predictor includes a final linear head as part of its output. Since our attention already projects our results to the embedding dimension, this acts as a simple linear layer that maintains the dimensions, though in other JEPA variants this layer may project down to the shared latent dimension.

To keep this layer simple, we'll add a consistent initialization of weights. 

In [115]:
mask_pred_head = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(mask_pred_head.weight, -0.1)
nn.init.zeros_(mask_pred_head.bias)
mask_pred_head.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [116]:
x = mask_pred_head(x)
x

tensor([[[0.5822, 0.5822, 0.5822, 0.5822, 0.5822, 0.5822],
         [0.5814, 0.5814, 0.5814, 0.5814, 0.5814, 0.5814],
         [0.5856, 0.5856, 0.5856, 0.5856, 0.5856, 0.5856],
         [0.5185, 0.5185, 0.5185, 0.5185, 0.5185, 0.5185],
         [0.5822, 0.5822, 0.5822, 0.5822, 0.5822, 0.5822],
         [0.5822, 0.5822, 0.5822, 0.5822, 0.5822, 0.5822],
         [0.5874, 0.5874, 0.5874, 0.5874, 0.5874, 0.5874],
         [0.5824, 0.5824, 0.5824, 0.5824, 0.5824, 0.5824],
         [0.5814, 0.5814, 0.5814, 0.5814, 0.5814, 0.5814]],

        [[0.5802, 0.5802, 0.5802, 0.5802, 0.5802, 0.5802],
         [0.5494, 0.5494, 0.5494, 0.5494, 0.5494, 0.5494],
         [0.5794, 0.5794, 0.5794, 0.5794, 0.5794, 0.5794],
         [0.5802, 0.5802, 0.5802, 0.5802, 0.5802, 0.5802],
         [0.5355, 0.5355, 0.5355, 0.5355, 0.5355, 0.5355],
         [0.4936, 0.4936, 0.4936, 0.4936, 0.4936, 0.4936],
         [0.5802, 0.5802, 0.5802, 0.5802, 0.5802, 0.5802],
         [0.5747, 0.5747, 0.5747, 0.5747, 0.5747, 0.57

### Predicted Latents vs Context Latents

By layering the extra masked predictor, the model can specialize these predicted latents for the reconstruction task without requiring the encoder's context latents to serve directly as predictions. Both tensors retain every gene position; the masks are applied later when we calculate the training loss. We'll walk through that selection in the training loss explainer notebook.

In [117]:
predicted_latent = x
predicted_latent

tensor([[[0.5822, 0.5822, 0.5822, 0.5822, 0.5822, 0.5822],
         [0.5814, 0.5814, 0.5814, 0.5814, 0.5814, 0.5814],
         [0.5856, 0.5856, 0.5856, 0.5856, 0.5856, 0.5856],
         [0.5185, 0.5185, 0.5185, 0.5185, 0.5185, 0.5185],
         [0.5822, 0.5822, 0.5822, 0.5822, 0.5822, 0.5822],
         [0.5822, 0.5822, 0.5822, 0.5822, 0.5822, 0.5822],
         [0.5874, 0.5874, 0.5874, 0.5874, 0.5874, 0.5874],
         [0.5824, 0.5824, 0.5824, 0.5824, 0.5824, 0.5824],
         [0.5814, 0.5814, 0.5814, 0.5814, 0.5814, 0.5814]],

        [[0.5802, 0.5802, 0.5802, 0.5802, 0.5802, 0.5802],
         [0.5494, 0.5494, 0.5494, 0.5494, 0.5494, 0.5494],
         [0.5794, 0.5794, 0.5794, 0.5794, 0.5794, 0.5794],
         [0.5802, 0.5802, 0.5802, 0.5802, 0.5802, 0.5802],
         [0.5355, 0.5355, 0.5355, 0.5355, 0.5355, 0.5355],
         [0.4936, 0.4936, 0.4936, 0.4936, 0.4936, 0.4936],
         [0.5802, 0.5802, 0.5802, 0.5802, 0.5802, 0.5802],
         [0.5747, 0.5747, 0.5747, 0.5747, 0.5747, 0.57

In [118]:
context_latent

tensor([[[-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
         [-1.0136, -1.0157, -0.9596, -0.9116, -1.0386, -1.0538],
         [ 0.8831,  0.9205,  1.0180,  1.1073,  1.0153,  1.0391],
         [-1.5923, -1.3612, -0.7581, -0.2064, -0.7748, -0.6279],
         [-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
         [-1.0063, -1.0072, -0.9826, -0.9615, -1.0172, -1.0239],
         [ 0.9362,  0.9568,  1.0106,  1.0598,  1.0091,  1.0222],
         [ 0.7995,  0.8626,  1.0271,  1.1777,  1.0226,  1.0627],
         [-1.0136, -1.0157, -0.9596, -0.9116, -1.0386, -1.0538]],

        [[-1.0076, -1.0080, -0.9827, -0.9610, -1.0166, -1.0228],
         [-1.4621, -1.2875, -0.8732, -0.4920, -0.8326, -0.7122],
         [-1.0167, -1.0177, -0.9594, -0.9093, -1.0376, -1.0520],
         [-1.0076, -1.0080, -0.9827, -0.9610, -1.0166, -1.0228],
         [ 0.4783,  0.6379,  1.0168,  1.3654,  1.0540,  1.1641],
         [-0.0514,  0.2442,  0.9456,  1.5908,  1.0143,  1.2182],
         [-1.0076, -1.0

# Latent Representation

We now have latent representations of our input cell states. These representations can then be used in a number of ways. You'll notice that the shape still contains the batch, our context length, which is the number of genes, and the embedding dimensions. Joint-embedding predictive architecture (JEPA) models work within this latent space. During our training loops we build energy-based loss functions to help steer and shape this space through backprop updates.